# Offline redocking collection controls

This notebook exercises public `dockingmt.summarize_redocking` over the retained
native evaluations from DockingMT #38. Original measurements, preparation limits
and provider identities remain historical; **no new Vina run occurs here**.

Keep 181L provisional preparation separate from the external/unassessed 1IEP
positive and displaced-domain controls. An explicitly labelled failure fixture
shows denominator handling; it is not an observed runtime failure. These selected
software controls do not establish a biological dataset success rate.

See [the contract](redocking_summary.md) and [owning issue #39](https://github.com/uibcdf/dockingmt/issues/39).
Execution uses the requested `molsyssuite@uibcdf_3.14` interpreter.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

assert "molsyssuite@uibcdf_3.14" in sys.executable
assert sys.version_info[:2] == (3, 14)
root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dockingmt/aggregation.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from devtools.qualify_redocking_summary import qualify

artifact = qualify()
destination = root / "devguide/validation/data/redocking_summary/summary.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(artifact, indent=2, allow_nan=False) + "\n")
print(json.dumps({"source": artifact["source"], "execution": artifact["execution"]}, indent=2))

{
  "source": {
    "path": "devguide/validation/data/redocking_evaluation/cases.json",
    "sha256": "383a3b1c75d66881d59862b0ff3e0676e1116b71530b72fe1af7d4c326870e1a",
    "size_bytes": 174825,
    "original_qualification": "dockingmt#38",
    "fresh_engine_execution": false
  },
  "execution": {
    "python": "3.14.7",
    "interpreter": "/home/diego/Myopt/miniconda3/envs/molsyssuite@uibcdf_3.14/bin/python",
    "dockingmt_version": "0.0.0+101.gbe39d74.dirty",
    "checkout_head": "308b406287dbd8df7f8eb7f5923bcb9ff5589f29",
    "implementation_sha256": "d033c3a8bbfbd4e44ffa34eaee6901f5a30efe95fa3b45e60d5d849ad1945ae5",
    "helper_sha256": "078a54303ca1f72e36e9de6f8ff97d2dc34b69c8c523fd306f39ec26d481c946"
  }
}


## Explicit observation denominators

The same 1IEP pair has one observed recovery and one observed nonrecovery. The
failure fixture remains unknown. Adding it changes the submitted denominator
from two to three while keeping the evaluated denominator at two. Cases with
no evaluated report infer no top-N policy.

In [2]:
rows = []
for cohort, summary in artifact["summaries"].items():
    for entry in summary["top_n"] or [None]:
        rows.append({
            "cohort": cohort,
            "submitted": summary["n_submitted"],
            "evaluated": summary["n_evaluated"],
            "failed": summary["n_failed"],
            "coverage": summary["evaluation_coverage"],
            "top_n": entry["requested"] if entry else None,
            "recovered": entry["n_recovered"] if entry else None,
            "not_recovered": entry["n_not_recovered"] if entry else None,
            "unknown": entry["n_unknown"] if entry else None,
            "fraction_evaluated": entry["fraction_of_evaluated"] if entry else None,
            "fraction_submitted": entry["fraction_of_submitted"] if entry else None,
        })
pd.DataFrame(rows)

,cohort,submitted,evaluated,failed,coverage,top_n,recovered,not_recovered,unknown,fraction_evaluated,fraction_submitted
0,181l_provisional,1,1,0,1.000000,1.0,1.0,0.0,0.0,1.0,1.000000
1,181l_provisional,1,1,0,1.000000,5.0,1.0,0.0,0.0,1.0,1.000000
2,1iep_external_controls,2,2,0,1.000000,1.0,1.0,1.0,0.0,0.5,0.500000
3,1iep_external_controls,2,2,0,1.000000,5.0,1.0,1.0,0.0,0.5,0.500000
4,1iep_with_declared_failure_fixture,3,2,1,0.666667,1.0,1.0,1.0,1.0,0.5,0.333333
5,1iep_with_declared_failure_fixture,3,2,1,0.666667,5.0,1.0,1.0,1.0,0.5,0.333333
6,all_failure_fixture,1,0,1,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
7,empty_collection,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Preserved cases and rejected policy mixing

The compact summary keeps case identities, original-report hashes, reference
declarations and context. It does not duplicate the full pose population or
reference coordinates. Keep the original JSON for replay and deeper audits.
Unknown preparation stays unknown; equality of declarations is an admission
rule, not scientific validation.

In [3]:
summaries = artifact["summaries"]
pair = summaries["1iep_external_controls"]
with_failure = summaries["1iep_with_declared_failure_fixture"]
assert pair["policy"]["preparation"]["partner"]["assessment"] == "unassessed"
assert all(entry["n_recovered"] == entry["n_not_recovered"] == 1 for entry in pair["top_n"])
assert all(entry["fraction_of_evaluated"] == 0.5 and entry["fraction_of_submitted"] == 1 / 3 for entry in with_failure["top_n"])
assert with_failure["cases"][-1]["context"]["observed_runtime_failure"] is False
assert summaries["all_failure_fixture"]["policy"] is None
assert summaries["empty_collection"]["evaluation_coverage"] is None
assert artifact["source"]["fresh_engine_execution"] is False
print("Rejected mixed 181L/1IEP declarations:", artifact["rejected_mixed_policy"]["reason"])
pd.DataFrame([
    {"case": case["case_id"], "poses": case["n_poses"],
     "first_rmsd_angstrom": case["first_pose"]["rmsd"],
     "closest_rmsd_angstrom": case["closest_pose"]["rmsd"],
     "original_report_sha256": case["evaluation_sha256"]}
    for case in pair["cases"]
])

Rejected mixed 181L/1IEP declarations: Invalid argument 'evaluations': Case '1iep_external': Incompatible comparison/method/protocol/preparation declarations; summarize separately.


,case,poses,first_rmsd_angstrom,closest_rmsd_angstrom,original_report_sha256
0,1iep_external,1,0.876036,0.876036,e2ae6478c0299212b21deb303076296d641966fce7c9f5...
1,1iep_displaced_domain,3,25.247243,22.243653,e7c19b71f8f66d3e4769550ff381a3ed5c54477e3090cb...
